In [1]:
import pandas as pd
import csv
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import f_oneway
from matplotlib.backends.backend_pdf import PdfPages

In [2]:
prefex_final_results = '/Users/rjing/Desktop/Machine_Learning_Nonpoint_Source_Pollution/paper_results/'

def cluster_analysis_and_weather_comparison(file_path, postfex, n_clusters=3):
    print(file_path.split('/')[-1].split('.')[0].split('_')[3:][0] + " " +
         '(' + file_path.split('/')[-1].split('.')[0].split('_')[3:][1] + 
          file_path.split('/')[-1].split('.')[0].split('_')[3:][2] +
         ')')
    os.environ["OMP_NUM_THREADS"] = "1"
    sns.set_style("darkgrid")  # Use fancy seaborn style for better visuals
    
    df = pd.read_csv(file_path)
    nutrient_features = ["NPK", "PK", "NK", "CK", "OF"]
    df = df.dropna(subset=nutrient_features)
    
    scaler = StandardScaler()
    df_scaled = scaler.fit_transform(df[nutrient_features])
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df["Cluster"] = kmeans.fit_predict(df_scaled)
    df.to_csv("clustered_water_sample.csv", index=False)
    
    print("\nCluster Counts:")
    print(df["Cluster"].value_counts())
    
    # Perform PCA for visualization
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(df_scaled)
    df["PCA1"] = pca_result[:, 0]
    df["PCA2"] = pca_result[:, 1]
    
    # Scatter Plot for Clusters
    cluster_colors = {0: "red", 1: "green", 2: "blue"}
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        x="PCA1", y="PCA2", hue="Cluster", palette=cluster_colors, 
        data=df, alpha=0.7, s=150, edgecolor="black"
    )
    plt.xlabel("Principal Component 1", fontsize=14)  # Increased font size for x label
    plt.ylabel("Principal Component 2", fontsize=14)  # Increased font size for y label
    plt.title("K-Means Clustering (PCA Reduced)" + " " + "-- " +
          file_path.split('/')[-1].split('.')[0].split('_')[3:][0], fontsize=14, fontweight='bold')
    
    plt.legend(title="Cluster", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.7)
    
    # Add a thinner box border to the plot
    ax = plt.gca()
    for _, spine in ax.spines.items():
        spine.set_edgecolor('black')
        spine.set_linewidth(0.8)  # Thinner border
    
    # Save scatter plot to the first folder
    os.makedirs(prefex_final_results + '/' + 'Cluster_Analysis/' + postfex + '/scatter_plots', exist_ok=True)
    plt.savefig(prefex_final_results + '/' + 'Cluster_Analysis/' + postfex + '/scatter_plots/' + file_path.split('/')[-1].split('.')[0] 
                + '_cluster_scatter_plot.png', bbox_inches='tight')
    plt.close()
    
    # Weather Feature Analysis
    weather_features = [
        "Dew Point Temperature (F)", "Visibility (mi)", "Average Wind Speed (knots)",
        "Maximum Sustained Wind Speed (knots)", "Maximum Gust (knots)", "Maximum Temperature (F)",
        "Minimum Temperature (F)", "Precipitation (in)"
    ]
    
    cluster_summary = df.groupby("Cluster")[weather_features].mean()
    print("\nWeather Condition Summary by Cluster:")
    print(cluster_summary)

    '''
    # Bar Plot for Weather Summary
    plt.figure(figsize=(16, 8))
    ax = cluster_summary.T.plot(
        kind="bar", figsize=(14, 7), color=["red", "green", "blue"], edgecolor="black", linewidth=0.8
    )
    plt.xlabel("Weather Variables", fontsize=14)  # Increased font size for x label
    plt.ylabel("Average Value", fontsize=14)  # Increased font size for y label
    plt.title("_Weather Conditions Across Clusters" + " " + "-- " +
          file_path.split('/')[-1].split('.')[0].split('_')[3:][0] + " " +
         '(' + file_path.split('/')[-1].split('.')[0].split('_')[3:][1] + " " +
          file_path.split('/')[-1].split('.')[0].split('_')[3:][2] + 
         ')', fontsize=14, fontweight='bold')
    
    plt.xticks(rotation=90, fontsize=14)
    plt.yticks(fontsize=14)
    plt.legend(title="Cluster", fontsize=14)
    plt.grid(axis="y", linestyle="--", alpha=0.7)  # Dashed grid for better readability
    plt.axhline(y=0, color='black', linewidth=1)
    # Add a thinner box border to the plot
    for _, spine in ax.spines.items():
        spine.set_edgecolor('black')
        spine.set_linewidth(0.8)  # Thinner border
    
    # Save bar plot to the second folder
    os.makedirs(prefex_final_results + '/' + 'Cluster_Analysis/' + postfex + '/weather_Conditions_Across_Clusters', exist_ok=True)
    plt.savefig(prefex_final_results + '/' + 'Cluster_Analysis/' + postfex + '/weather_Conditions_Across_Clusters/' 
                + file_path.split('/')[-1].split('.')[0] 
                + 'weather_bar_plot.png', bbox_inches='tight')
    plt.close()
    '''
    # ✅ save weather_summary_table
    weather_table_dir = os.path.join(prefex_final_results, "Cluster_Analysis", postfex, "weather_summary_tables")
    os.makedirs(weather_table_dir, exist_ok=True)
    
    weather_summary_table_path = os.path.join(
        weather_table_dir, file_path.split('/')[-1].replace(".csv", "_weather_summary.csv")
    )
    cluster_summary.reset_index().to_csv(weather_summary_table_path, index=False)
    print(f"✅ save weather_summary_table to：{weather_summary_table_path}")

    
    # ANOVA Analysis for Weather Features
    print("\nStatistical Comparison (ANOVA p-values):")
    for feature in weather_features:
        groups = [df[df["Cluster"] == c][feature].dropna() for c in df["Cluster"].unique()]
        stat, p = f_oneway(*groups)
        print(f"{feature}: p-value = {p:.4f}")
    
    return df

In [3]:
#prefex = '/Users/rjing/Desktop/Machine_Learning_Nonpoint_Source_Pollution/data/water_runoff_sample/Total_Phosphorus/'
prefex = '/Users/rjing/Desktop/Machine_Learning_Nonpoint_Source_Pollution/data/water_runoff_sample/Total_Nitrogen/'
#prefex = '/Users/rjing/Desktop/Machine_Learning_Nonpoint_Source_Pollution/data/water_runoff_sample/Ammoniacal_Nitrogen/'
#prefex = '/Users/rjing/Desktop/Machine_Learning_Nonpoint_Source_Pollution/data/water_runoff_sample/Dissolved_Phosphorus/'
#prefex = '/Users/rjing/Desktop/Machine_Learning_Nonpoint_Source_Pollution/data/water_runoff_sample/Nitrate_Nitrogen/'
#prefex = '/Users/rjing/Desktop/Machine_Learning_Nonpoint_Source_Pollution/data/water_runoff_sample/Particulate_Phosphorus/'

file_list = [f for f in os.listdir(prefex) if f.endswith(".csv")]
print(file_list)

selected_files = []
# Filter files that match the "Total_Nitrogen" condition
for file in file_list:
    df_clustered = cluster_analysis_and_weather_comparison(prefex + file, prefex.split('/')[-2] , n_clusters=3)


print("Job done!")

['Water_sample_runoff_Citrus_Total_Nitrogen.csv', 'Water_sample_runoff_Corn_Total_Nitrogen.csv', 'Water_sample_runoff_Rice_Total_Nitrogen.csv', 'Water_sample_runoff_Vegetables_Total_Nitrogen.csv']
Citrus (TotalNitrogen)

Cluster Counts:
Cluster
0    38
1    19
2    15
Name: count, dtype: int64


C:\Users\rjing\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1440: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



Weather Condition Summary by Cluster:
         Dew Point Temperature (F)  Visibility (mi)  \
Cluster                                               
0                         0.043629         0.207216   
1                        -0.284335         0.228268   
2                         0.319983        -0.027973   

         Average Wind Speed (knots)  Maximum Sustained Wind Speed (knots)  \
Cluster                                                                     
0                          0.285716                              0.287517   
1                          0.613945                              0.174806   
2                          0.192874                             -0.004149   

         Maximum Gust (knots)  Maximum Temperature (F)  \
Cluster                                                  
0                    0.094445                -0.056838   
1                    0.361426                -0.101775   
2                    0.298440                 0.176386   

        

C:\Users\rjing\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1440: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



Weather Condition Summary by Cluster:
         Dew Point Temperature (F)  Visibility (mi)  \
Cluster                                               
0                         0.471686         0.114170   
1                         0.245728         0.293932   
2                         0.215242        -0.133116   

         Average Wind Speed (knots)  Maximum Sustained Wind Speed (knots)  \
Cluster                                                                     
0                          0.094099                              0.048978   
1                          0.248614                              0.542303   
2                         -0.000184                              0.071993   

         Maximum Gust (knots)  Maximum Temperature (F)  \
Cluster                                                  
0                    0.095927                 0.456274   
1                   -0.736664                 0.435553   
2                    0.294438                 0.219742   

        

C:\Users\rjing\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1440: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



Weather Condition Summary by Cluster:
         Dew Point Temperature (F)  Visibility (mi)  \
Cluster                                               
0                         0.491998         0.189170   
1                         0.112369         0.466094   
2                         0.136753         0.242952   

         Average Wind Speed (knots)  Maximum Sustained Wind Speed (knots)  \
Cluster                                                                     
0                          0.483199                              0.325617   
1                         -0.090619                             -0.206875   
2                          0.334442                              0.399922   

         Maximum Gust (knots)  Maximum Temperature (F)  \
Cluster                                                  
0                    0.005966                 0.509869   
1                    0.325139                 0.026122   
2                   -0.258843                 0.292096   

        

C:\Users\rjing\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1440: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



Weather Condition Summary by Cluster:
         Dew Point Temperature (F)  Visibility (mi)  \
Cluster                                               
0                        -0.269483        -0.104708   
1                        -0.211384         0.121143   
2                        -0.755855        -0.293947   

         Average Wind Speed (knots)  Maximum Sustained Wind Speed (knots)  \
Cluster                                                                     
0                          0.093572                             -0.152776   
1                          0.382109                              0.260989   
2                          0.320450                              0.425195   

         Maximum Gust (knots)  Maximum Temperature (F)  \
Cluster                                                  
0                    0.091985                -0.263185   
1                   -0.335262                -0.198956   
2                    0.553492                -0.652750   

        